Call LLM

In [ ]:
from langchain_community.llms import  Ollama

llm = Ollama(model="llama2")
response = llm.invoke("What is the capital of France?")
print(response)


The capital of France is Paris.


In [28]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(response)

'\nThe capital of France is Paris.'

Simple Chain

In [26]:
chain = llm | output_parser
chain.invoke("What is the capital of France?")

'\nThe capital of France is Paris.'

Structure Output

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class MobileReview(BaseModel):
    phone_model: str = Field( description="The model of the mobile phone ")
    rating : str = Field( description="The rating of the mobile phone out of 5 ")
    pros : str = Field( description="The pros of the mobile phone ")
    cons : str = Field( description="The cons of the mobile phone ")
    summary : str = Field( description="A brief summary of the mobile phone review ") 



review_txt = """
The Samsung Galaxy S21 FE 5G feels like a phone made for people who want a premium experience without paying flagship prices. The moment you hold it, it gives a solid, comfortable vibe — not too heavy, not too light, and the matte finish makes it look clean and modern. The display is one of the best parts of the phone. It looks sharp, bright, and smooth, whether you're watching videos, scrolling Instagram, chatting, or just browsing. Even outdoors, the brightness holds up well, and the colors pop nicely without feeling overly artificial.
Performance-wise, the phone runs smoothly for almost everything you throw at it. Everyday tasks like switching between apps, replying to messages, watching reels, and even playing moderate games feel fluid and responsive. It handles multitasking well, though during very heavy gaming sessions or long performance-heavy tasks, you might feel a bit of heat or slight dips — but for normal use, it’s more than enough.
The camera setup is reliable and gives good results for day-to-day photography. It’s great for social media pictures, portraits, and general outdoor shots. In low light, it still performs decently, though you can see it’s not on the same level as top flagship cameras. But for Instagram, Snapchat, or casual everyday pics, it’s absolutely fine and delivers natural colors.

"""


NotImplementedError: 

Prompt Template 

In [29]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me short joke about {topic}.")


In [33]:
chain = prompt | llm
chain.invoke({"topic": "dragons"})

"\nSure! Here's a quick dragon joke for you:\n\nWhy did the dragon go to the dentist?\n\nBecause he had a tooth-tauristic problem!"

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import  Ollama
from langchain_core.prompts import ChatPromptTemplate


# Define the Prompt 

prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")

llm = Ollama(model="llama2")

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

result = chain.invoke({"topic": "computers"})
print(result)

LLM Messages

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama   # IMPORTANT

FAISS_DB_PATH = "faiss_selfhelp_db"

llm = ChatOllama(model="llama2")

# Load embeddings + FAISS
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(FAISS_DB_PATH, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})




def get_context(query):
    docs = retriever.invoke(query)
    print(f"Retrieved {len(docs)} documents from vectorstore.", docs)
    return "\n\n---\n\n".join(
        [f"From 『{doc.metadata.get('book','Unknown Book')}』\n{doc.page_content}" for doc in docs]
    )


question = "How can I build better habits?"
context = get_context(question)


prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert self-help coach. "
     "Use ONLY the context to answer. "
     "Be friendly, clear, and motivating. "
     "If the answer is missing in context, reply: "
     "'I don't have information about that in my current books.'"
    ),
    ("human",
     "Context:\n{context}\n\n"
     "Question:\n{question}\n\n"
     "Answer:" 
    )
])



chain = prompt | llm   # NO RETRIEVER HERE

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)


/Users/tusharkungar/Desktop/Self_Help_RAG_App /myenv/lib/python3.14/site-packages/langsmith/schemas.py:24: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import (
/Users/tusharkungar/Desktop/Self_Help_RAG_App /myenv/lib/python3.14/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Retrieved 3 documents from vectorstore. [Document(metadata={'source': 'books/Atomic habits ( PDFDrive ).pdf', 'file_path': 'books/Atomic habits ( PDFDrive ).pdf', 'page': 104, 'total_pages': 256, 'format': 'PDF 1.4', 'title': 'Atomic habits \\( PDFDrive.com \\).pdf', 'author': 'James Clear', 'subject': '', 'keywords': '', 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]', 'producer': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creationDate': "D:20200430184622+00'00'", 'modDate': '', 'trapped': '', 'book': 'Atomic habits ( PDFDrive )'}, page_content='Your habits are modern-day solutions to ancient desires. New versions of old\nvices. The underlying motives behind human behavior remain the same. The\nspecific habits we perform differ based on the period of history.\nHere’s the powerful part: there are many different ways to address the same\nunderlying motive. One person might learn to reduce stress by smoking a\ncigarette. Another person learns to ease their anxiety by going fo

RAG Chains 

In [2]:
from langchain_core.prompts import ChatPromptTemplate
template = """Answere the question based only on the following context:{context}
Question: {question}
Answere : 
"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:



from langchain.schema.runnable import RunnablePassthrough

rag_chain = (
    {"context" : retriever, "question" : RunnablePassthrough()} | prompt 
)
results = rag_chain.invoke("What is the habbit")

print(results)

messages=[HumanMessage(content='Answere the question based only on the following context:[Document(metadata={\'source\': \'books/How To Win Friends And Influence People - Carnegie, Dale.pdf\', \'file_path\': \'books/How To Win Friends And Influence People - Carnegie, Dale.pdf\', \'page\': 96, \'total_pages\': 232, \'format\': \'PDF 1.4\', \'title\': \'How To Win Friends And Influence People\', \'author\': \'Carnegie, Dale\', \'subject\': \'\', \'keywords\': \'\', \'creator\': \'calibre (0.9.35) [http://calibre-ebook.com]\', \'producer\': \'calibre (0.9.35) [http://calibre-ebook.com]\', \'creationDate\': "D:20230208193236+00\'00\'", \'modDate\': "D:20230208193237+00\'00\'", \'trapped\': \'\', \'book\': \'How To Win Friends And Influence People - Carnegie, Dale\'}, page_content=\'‘ “I don’t know what you did to the old boy,” the steward greeted me,\\n“but he sure is sold on you!”\\n‘Think of it! I had been drumming at that man for four years – trying\\nto get his business – and I’d still

In [4]:
def doc2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [11]:
rag_chain = (
    {"context" : retriever | doc2str , "question" : RunnablePassthrough()}
    | prompt
    | llm 
)
question = "How to control emotion and improve habit"
response = rag_chain.invoke(question)
print(response.content)

The answer to the question "How to control emotion and improve habit" is not explicitly mentioned in the provided text. However, based on the context and the information provided, there are some suggestions that can be made:

1. Become aware of your habits: The first step to controlling emotions and improving habits is to become aware of them. Use the Habits Scorecard to identify your current habits and understand how they are associated with certain emotions or situations.
2. Use implementation intentions: Once you are aware of your habits, use implementation intentions to consciously decide when and where you want to perform a particular habit. For example, if you want to improve your breathing and smiling routine, decide when and where you want to practice it, such as when you feel stressed at work or sad about life.
3. Stack new habits on top of existing ones: Use habit stacking to build new habits on top of existing ones. For example, if you want to improve your breathing and smil

Conversational RAG

In [12]:
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []

chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response.content),
    
])

print(chat_history)

[HumanMessage(content='How to control emotion and improve habit', additional_kwargs={}, response_metadata={}), AIMessage(content='The answer to the question "How to control emotion and improve habit" is not explicitly mentioned in the provided text. However, based on the context and the information provided, there are some suggestions that can be made:\n\n1. Become aware of your habits: The first step to controlling emotions and improving habits is to become aware of them. Use the Habits Scorecard to identify your current habits and understand how they are associated with certain emotions or situations.\n2. Use implementation intentions: Once you are aware of your habits, use implementation intentions to consciously decide when and where you want to perform a particular habit. For example, if you want to improve your breathing and smiling routine, decide when and where you want to practice it, such as when you feel stressed at work or sad about life.\n3. Stack new habits on top of exis

In [14]:
from langchain_core.prompts import MessagesPlaceholder
contexualize_q_system_prompt = (
    "Given a chat history and the latest user question"
    "which might reference context in the chat history, "
    "rephrase the user question to be a standalone question, "
    "Without the chathistory. Do not the answer the question, "
    "just rephrase it. if no needed and otherwise, return the question as is."
)

contexualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contexualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

contexualize_chain = contexualize_q_prompt | llm 
contexualize_chain.invoke({"input": "How to be aware mediation", "chat_history": chat_history})

AIMessage(content='The question "How to be aware meditation" is a bit vague, but here are some suggestions based on the context provided:\n\n1. Start with short sessions: Begin with short meditation sessions, such as 5-10 minutes, and gradually increase the duration as you become more comfortable with the practice.\n2. Focus on your breath: Bring your attention to your breath, feeling the sensation of the air entering and leaving your nostrils. When your mind wanders, gently bring it back to your breath without judgment.\n3. Observe your thoughts: Notice any thoughts or emotions that arise during meditation without getting caught up in them. Practice a non-judgmental attitude towards your thoughts and emotions.\n4. Use guided meditations: Listen to guided meditations, either through recordings or with the help of a meditation app. These can provide a structured framework for your meditation practice and help you stay focused.\n5. Make it a habit: Incorporate meditation into your daily 

In [15]:
from langchain.chains import create_history_aware_retriever 

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contexualize_q_prompt
)

history_aware_retriever.invoke({"input" : "Smiling and breating routing ", "chat_history": chat_history})

[Document(metadata={'source': 'books/How To Win Friends And Influence People - Carnegie, Dale.pdf', 'file_path': 'books/How To Win Friends And Influence People - Carnegie, Dale.pdf', 'page': 75, 'total_pages': 232, 'format': 'PDF 1.4', 'title': 'How To Win Friends And Influence People', 'author': 'Carnegie, Dale', 'subject': '', 'keywords': '', 'creator': 'calibre (0.9.35) [http://calibre-ebook.com]', 'producer': 'calibre (0.9.35) [http://calibre-ebook.com]', 'creationDate': "D:20230208193236+00'00'", 'modDate': "D:20230208193237+00'00'", 'trapped': '', 'book': 'How To Win Friends And Influence People - Carnegie, Dale'}, page_content='it:\n‘Action seems to follow feeling, but really action and feeling go\ntogether; and by regulating the action, which is under the more direct\ncontrol of the will, we can indirectly regulate the feeling, which is not.\n‘Thus the sovereign voluntary path to cheerfulness, if our cheerfulness\nbe lost, is to sit up cheerfully and to act and speak as if chee

In [16]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain 

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert self-help coach assistant. Use the following context to answer the user's questions. "),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

question_answering_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answering_chain)

In [17]:
rag_chain.invoke({"input" : "How to improve focus during work?", "chat_history": chat_history})

{'input': 'How to improve focus during work?',
 'chat_history': [HumanMessage(content='How to control emotion and improve habit', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The answer to the question "How to control emotion and improve habit" is not explicitly mentioned in the provided text. However, based on the context and the information provided, there are some suggestions that can be made:\n\n1. Become aware of your habits: The first step to controlling emotions and improving habits is to become aware of them. Use the Habits Scorecard to identify your current habits and understand how they are associated with certain emotions or situations.\n2. Use implementation intentions: Once you are aware of your habits, use implementation intentions to consciously decide when and where you want to perform a particular habit. For example, if you want to improve your breathing and smiling routine, decide when and where you want to practice it, such as when you feel stres

Normal Code to the Production Ready App

In [26]:
import sqlite3
import json
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn 

def create_application_logs():
    conn = get_db_connection()
    conn.execute(''' CREATE TABLE IF NOT EXISTS application_logs (id INTEGER PRIMARY KEY AUTOINCREMENT, 
                session_id TEXT,
                user_query TEXT,
                gpt_response TEXT,
                model TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)
''')
    
    conn.close()


def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()

    # Convert dict → JSON string
    gpt_response_str = json.dumps(gpt_response)

    conn.execute(
        '''
        INSERT INTO application_logs 
        (session_id, user_query, gpt_response, model) 
        VALUES (?, ?, ?, ?)
        ''',
        (session_id, user_query, gpt_response_str, model)
    )

    conn.commit()
    conn.close()


def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    conn.execute(' SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at ASC',
                        (session_id,)).fetchall()
    messages = []

    for row in cursor.fetchall():
        messages.extend([
           {"role" : "human", "content" : row["user_query"]},
           {"role" : "ai", "content" : row["gpt_response"]}
        ])
    conn.close()
    return messages

create_application_logs()

In [27]:
import uuid 
session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)

question1 = "When was GreenGrow innovation founded?"
answer1 = rag_chain.invoke({"input" : question1, "chat_history": chat_history})
model = "llama2"
insert_application_logs(session_id, question1, answer1, model)
print(f"Human: {question1}\nAI: {answer1}\n")

[]


TypeError: Object of type Document is not JSON serializable

NEW USER

In [28]:
session_id = str(uuid.uuid4())
question = "What are some of their key products?"
chat_history = get_chat_history(session_id)
print(chat_history)

answere = rag_chain.invoke({"input" : question, "chat_history": chat_history})['answere']
insert_application_logs(session_id, question, answere, 'gpt-3.5-turbo')
print(f"Human: {question}\nAI: {answere}\n")

[]


KeyError: 'answere'